# NB01: Data Collection

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250093214


## Setup

Run the cell below to ensure all required packages are installed before running all other cells.

In [1]:
import json
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## The question

Power creep is defined as "the strengthening of [a] game and its pieces over time possibly to the point where new pieces invalidate older ones" [(Magruder, 2022)](https://journals.sagepub.com/doi/full/10.1177/15554120211050812#bibr20-15554120211050812). This occurs because developers want to keep their games fresh and exciting in order to keep players coming back [(Magic the Gathering on new cards sets)](https://magic.wizards.com/en/news/card-preview/fire-it-2019-06-21). However, too much power creep can make a game unrecognizeable. One card game that is perhaps infamous for power creep is Yu-Gi-Oh. One search on Reddit will provide many posts from both dedicated Yu-Gi-Oh subreddits and other card game subreddits discussing the power creep in Yu-Gi-Oh. Some posts are as old as five years, some as recent as two months ago, showing how pervasive and long-lasting the issue of power creep is. (Re-watch RTGame Yu-Gi-Oh video about coming back to the game). 

Yu-Gi-Oh has been around for nearly 30 years, and though power creep is an issue, it has still been going strong. Another franchise that has been going arguably stronger for the same length of time is Pokemon. It is so beloved that parents that grew up with Pokemon are now sharing it with their own children [(Amenabar, 2022)](https://www.washingtonpost.com/video-games/2022/08/10/pokemon-starter-parents-kids/). Yu-Gi-Oh has not experienced the same rite-of-passage experience as Pokemon has, in part due to how complicated the game is [(short Reddit thread about the subject)](https://www.reddit.com/r/Yugioh101/comments/onn3vm/dear_parents_do_your_kids_play_yugioh_and_is_it/). If Pokemon experiences power creep in the same way, this parental bonding method might not be as effective any more. Additionally, fans could get tired of having to buy all of the new games and DLC just to get access to all of the new, strong Pokemon. However, this would only occur if Pokemon experiences a high level of power creep.

Question: Has Pokemon experienced power creep over its thirty years of existance? If so, to what extent is power creep experienced?


## Where is the data coming from to answer this question?

All data collected to answer this question comes from [PokeAPI](https://pokeapi.co/docs/v2). This is a robust API offering endpoints for many different kinds of Pokemon data, but the three I will be collecting data on are their [Generation API](https://pokeapi.co/docs/v2#games-section), their [Pokemon Species API](https://pokeapi.co/docs/v2#pokemon-species), and their [Pokemon API](https://pokeapi.co/docs/v2#pokemon).

Suppelementary data will be retrieved from either [Bulbapedia](https://bulbapedia.bulbagarden.net/wiki/Main_Page), [Pokemon Database](https://pokemondb.net/), or [Serebii.net](https://www.serebii.net/).


## Getting a list of all Pokemon

First, I am going to use PokeAPI's Generation API to collect a list of all of the Pokemon species introduced in every generation. A species, as defined by PokeAPI, "forms the basis for at least one Pokemon." For instance, take the Pokemon Wormadam. Wormadam has three forms based on its cloak: Plant Cloak, Sandy Cloak, and Trash Cloak, but Wormadam is the underlying species of all three forms.

In [2]:
#Initializing empty lists for storage
api_calls = []
mon_name = []
gen_list = []
gen_mon = {}

#Calling the Generation API
for i in range(1,10):
    with open(f'../data/raw/pokemon_list_gen_{i}.json', mode='w') as f:
        request = requests.get(f"https://pokeapi.co/api/v2/generation/{i}")
        pokemon_json = request.json()
        json.dump(pokemon_json,f,indent=4)
    with open(f'../data/raw/pokemon_list_gen_{i}.json', mode='r') as f:
        data = json.load(f)
        df = pd.json_normalize(
            data,
            record_path= 'pokemon_species',
            meta = 'name',
            record_prefix= 'pokemon_'
        )

#Storing the API call URL and the Pokemon names for use in later API calls
    for i in range(len(df['pokemon_url'])):
        api_calls.append(df['pokemon_url'][i])
        mon_name.append(df['pokemon_name'][i])

#Creating a generation list for later usage in renaming columns
    gen_list.append(df['name'][0])

#Creating a dictionary of every Pokemon added in each Generation for later usage in assigning Generation to a dataframe
    for name in df['pokemon_name']:
        if df['name'][0] not in gen_mon.keys():
            gen_mon[df['name'][0]] = [name]
        else:
            gen_mon[df['name'][0]].append(name)


Now that I have a list of all the different Pokemon species' APIs, I am going to call each species' API to get the API calls for each Pokemon form individually (ex: all three types of Wormadam cloaks rather than just the Wormadam species as a whole). But, first, I am going to create a function to perform the remaining API calls for me.

### Pokemon Species Data

In [3]:
def call_api(call_list,name_list,file_suffix):
##Takes in a list of APIs to call, a list of names for the files, and a suffix for the file to call and store .json data from an API##
    for i in range(len(call_list)):
        with open(f'../data/raw/{name_list[i]}_{file_suffix}.json', mode='w') as f:
            request = requests.get(call_list[i])
            pokemon_json = request.json()
            json.dump(pokemon_json,f,indent=4)

In [4]:
call_api(api_calls,mon_name,'spec')

In [5]:
#Initializing empty lists for storage
stat_calls = []
form_name = []

#Getting the list of each Pokemon form from API alongside calls
for i in range(len(api_calls)):
        with open(f'../data/raw/{mon_name[i]}_spec.json', mode='r') as f:
                data = json.load(f)
                df = pd.json_normalize(
                data,
                record_path = 'varieties',
                sep = '_'
                )

        for i in range(len(df['pokemon_url'])):
                stat_calls.append(df['pokemon_url'][i])
                form_name.append(df['pokemon_name'][i])


### Pokemon Stats Data

In [6]:
call_api(stat_calls,form_name,'stat')

In [7]:
print(api_calls)

['https://pokeapi.co/api/v2/pokemon-species/1/', 'https://pokeapi.co/api/v2/pokemon-species/4/', 'https://pokeapi.co/api/v2/pokemon-species/7/', 'https://pokeapi.co/api/v2/pokemon-species/10/', 'https://pokeapi.co/api/v2/pokemon-species/13/', 'https://pokeapi.co/api/v2/pokemon-species/16/', 'https://pokeapi.co/api/v2/pokemon-species/19/', 'https://pokeapi.co/api/v2/pokemon-species/21/', 'https://pokeapi.co/api/v2/pokemon-species/23/', 'https://pokeapi.co/api/v2/pokemon-species/27/', 'https://pokeapi.co/api/v2/pokemon-species/29/', 'https://pokeapi.co/api/v2/pokemon-species/32/', 'https://pokeapi.co/api/v2/pokemon-species/37/', 'https://pokeapi.co/api/v2/pokemon-species/41/', 'https://pokeapi.co/api/v2/pokemon-species/43/', 'https://pokeapi.co/api/v2/pokemon-species/46/', 'https://pokeapi.co/api/v2/pokemon-species/48/', 'https://pokeapi.co/api/v2/pokemon-species/50/', 'https://pokeapi.co/api/v2/pokemon-species/52/', 'https://pokeapi.co/api/v2/pokemon-species/54/', 'https://pokeapi.co/ap

All of the necessary calls have been made, and all the data stored in the 'raw' folder under 'data'.

## Storing the collected data in dataframes

Now that the data has been collected and safely stored, it is time to arrange the data into neat dataframes.

### How many dataframes will be created?
There are going to be two different dataframes: one containing the species data of the Pokemon, and one containing the actual stats data of the Pokemon.

### What will each dataframe contain?
For the dataframe with the species data, each row is going to contain the ID of the Pokemon, the name of the Pokemon, the flags of whether the Pokemon is a baby, a legendary, or a mythical, and the Generation that the Pokemon is associated with.

For the dataframe with the stats data, each row is going to contain the ID, name of the Pokemon, and its associated Generation once again, and all of their individual base stats and the names of those stats.

### Why two different dataframes?

There are a couple of reasons why two different dataframes are needed:
- Having the species and the individual stats data separated out will make for easier analysis later, as I can choose to combine the dataframes with pd.merge() if the need arises, or I can keep them separate and perform individual analyses on the data they contain
- The dataframe containing the stats data has six rows for each Pokemon (one for each kind of base stat), while the dataframe containing the species data has one row for each Pokemon, so attempting to combine them out of the gate will lead to misaligned dataframes and bad data
- The data is stored in two different API calls, so creating two different dataframes is easier for me, personally

### Why am I creating and storing the dataframes here? Why not in NB02?

I have already stored all of the API calls for both species and stats data in this notebook, and variables do not carry over between notebooks. As such, to avoid repeating code as much as possible, I am going to create the dataframes here.

In [ ]:
#Creating the dataframe with species data
spec_df = ''
with open(f'../data/raw/{mon_name[0]}_spec.json', mode='r') as f:
        data = json.load(f)
        df = pd.json_normalize(
            data,
            record_path = 'varieties',
            meta = ['id', 'name', 'is_baby', 'is_legendary', 'is_mythical'],
            sep='_'
        )
        gen_df = pd.DataFrame(data['generation'],index=[0]).rename(columns={'name':'gen', 'url':'gen_url'})

initial_df = pd.concat([df,gen_df],axis=1)


for i in range(1, len(api_calls)):
        with open(f'../data/raw/{mon_name[i]}_spec.json', mode='r') as f:
                data = json.load(f)
                df = pd.json_normalize(
                        data,
                        record_path = 'varieties',
                        meta = ['id', 'name', 'is_baby', 'is_legendary', 'is_mythical'],
                        sep='_'
                )
                gen_df = pd.DataFrame(data['generation'],index=[0]).rename(columns={'name':'gen', 'url':'gen_url'})
        dfc = pd.concat([df,gen_df],axis=1)
        spec_df = pd.concat([initial_df,dfc],axis=0, ignore_index = True)
        initial_df = spec_df

,is_default,pokemon_name,pokemon_url,id,name,is_baby,is_legendary,is_mythical,gen,gen_url
0,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/
1,True,charmander,https://pokeapi.co/api/v2/pokemon/4/,4,charmander,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/
2,True,squirtle,https://pokeapi.co/api/v2/pokemon/7/,7,squirtle,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/
3,True,caterpie,https://pokeapi.co/api/v2/pokemon/10/,10,caterpie,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/
4,True,weedle,https://pokeapi.co/api/v2/pokemon/13/,13,weedle,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/
...,...,...,...,...,...,...,...,...,...,...
1346,True,arboliva,https://pokeapi.co/api/v2/pokemon/930/,930,arboliva,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/
1347,True,revavroom,https://pokeapi.co/api/v2/pokemon/966/,966,revavroom,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/
1348,True,arctibax,https://pokeapi.co/api/v2/pokemon/997/,997,arctibax,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/
1349,True,baxcalibur,https://pokeapi.co/api/v2/pokemon/998/,998,baxcalibur,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/


The use of .rename(): When I was creating the dataframe, I found that I had to add Generation separately due to the way it was stored in the API and then concatenate it onto the dataframe by row, but when I tried to concatenate them originally, I ran into name errors since the name of the Generation and the URL to call the generation were under the same name as the column storing the Pokemon name and the URL to get data about the Pokemon themselves. As such, I used .rename() to change the conflicting names of columns to something else, passing a dictionary with the original column names as keys and my new column names as values.

In [ ]:
## Creating the dataframe with stats data
with open(f'../data/raw/{form_name[0]}_stat.json', mode='r') as f:
    data = json.load(f)
    initial_df = pd.json_normalize(
        data,
        record_path = 'stats',
        meta = ['id', 'name'],
        sep='_'
    )

for i in range(1, len(stat_calls)):
    with open(f'../data/raw/{form_name[i]}_stat.json', mode='r') as f:
            data = json.load(f)
            df = pd.json_normalize(
                data,
                record_path = 'stats',
                meta = ['id','name'],
                sep='_'
            )
    stats_df = pd.concat([initial_df,df],axis=0, ignore_index = True)
    initial_df = stats_df

,base_stat,effort,stat_name,stat_url,id,name
0,45,0,hp,https://pokeapi.co/api/v2/stat/1/,1,bulbasaur
1,49,0,attack,https://pokeapi.co/api/v2/stat/2/,1,bulbasaur
2,49,0,defense,https://pokeapi.co/api/v2/stat/3/,1,bulbasaur
3,65,1,special-attack,https://pokeapi.co/api/v2/stat/4/,1,bulbasaur
4,65,0,special-defense,https://pokeapi.co/api/v2/stat/5/,1,bulbasaur
...,...,...,...,...,...,...
8101,175,0,attack,https://pokeapi.co/api/v2/stat/2/,10325,baxcalibur-mega
8102,117,0,defense,https://pokeapi.co/api/v2/stat/3/,10325,baxcalibur-mega
8103,105,0,special-attack,https://pokeapi.co/api/v2/stat/4/,10325,baxcalibur-mega
8104,101,0,special-defense,https://pokeapi.co/api/v2/stat/5/,10325,baxcalibur-mega


Now that the two dataframes have been created, it is time to save them as csv files for usage in later notebooks.

In [20]:
spec_df.to_csv('../data/processed/spec_df.csv')

stats_df.to_csv('../data/processed/stats_df.csv')